# Prompt Management Lifecycle

This notebook walks through the core phases of prompt management:

1. **Versioning** — Register and track prompt versions
2. **A/B Comparison** — Run the same test cases against different prompt versions
3. **Evaluation** — Measure output quality with DeepEval metrics
4. **Reporting** — Generate a comparison report

**Prerequisites**: Set your provider in the config cell below and ensure `OPENAI_API_KEY` (or equivalent) is in your `.env` file.

In [ ]:
# ── Config ──────────────────────────────────────────────────────────
PROVIDER = "openai"        # "openai" | "anthropic" | "ollama"
MODEL    = None             # None = provider default, or e.g. "gpt-4o"
TEMPERATURE = 0.3           # Low temperature for consistent evaluation
# ───────────────────────────────────────────────────────────────────

## 1. Prompt Versioning

Every prompt change gets a version. We track:
- **Content**: The prompt text
- **Metadata**: Version, author, timestamp, notes
- **Status**: draft → staging → production → retired

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional


@dataclass
class PromptVersion:
    """A single versioned prompt entry."""
    version: str
    content: str
    system_prompt: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    author: str = "auto"
    notes: str = ""
    status: str = "draft"


# Register two versions of a classification prompt
v1 = PromptVersion(
    version="1.0",
    content="Classify this customer message into one category: billing, technical, general. Return only the category name.",
    author="alice",
    notes="Initial version — basic 3-category classifier",
)

v2 = PromptVersion(
    version="2.0",
    content=(
        "You are a customer support classifier.\n\n"
        "Classify the customer message below into exactly one category:\n"
        "- billing (payment issues, invoices, refunds)\n"
        "- technical (bugs, errors, setup problems)\n"
        "- general (everything else)\n\n"
        "Customer message: {input}\n\n"
        "Respond with ONLY the category name, nothing else."
    ),
    author="bob",
    notes="Rewrite with block structure and explicit category definitions",
)

print(f"Registered v{v1.version}: {v1.notes}")
print(f"Registered v{v2.version}: {v2.notes}")
print(f"\nv1.0 is {len(v1.content)} chars, v2.0 is {len(v2.content)} chars")
print(f"v2.0 adds {len(v2.content) - len(v1.content)} chars of structured instruction")

## 2. A/B Comparison

Run the same test inputs against both prompt versions and compare outputs.
This is the core of prompt evaluation — **same input, different prompt, measurable difference**.

In [ ]:
import sys
sys.path.insert(0, "../..")  # Access utils/ from repo root

from utils.llm_client import call_llm


def run_prompt(prompt_template: str, test_inputs: list[dict], provider: str, model=None) -> list[dict]:
    """Run a prompt against test inputs and return results."""
    results = []
    for case in test_inputs:
        user_input = case["input"]
        if "{input}" in prompt_template:
            filled = prompt_template.replace("{input}", user_input)
        else:
            filled = f"{prompt_template}\n\n{user_input}"

        actual = call_llm(
            prompt=filled,
            provider=provider,
            model=model,
            temperature=TEMPERATURE,
            max_tokens=100,
        )
        results.append({
            "input": user_input,
            "actual_output": actual.strip(),
            "expected_output": case.get("expected_output", ""),
        })
    return results


# Test cases — diverse customer support messages
test_cases = [
    {"input": "I was charged twice for my subscription", "expected_output": "billing"},
    {"input": "The app keeps crashing when I open settings", "expected_output": "technical"},
    {"input": "What are your business hours?", "expected_output": "general"},
    {"input": "I need a refund for my last payment", "expected_output": "billing"},
    {"input": "How do I reset my password?", "expected_output": "technical"},
]

print(f"Running {len(test_cases)} test cases against both versions...\n")

# Run v1.0
print("── v1.0 (basic classifier) ──")
results_v1 = run_prompt(v1.content, test_cases, PROVIDER, MODEL)
for r in results_v1:
    match = "✓" if r["actual_output"].lower() == r["expected_output"].lower() else "✗"
    print(f"  {match} '{r['input'][:40]}...' → '{r['actual_output']}'")

print()

# Run v2.0
print("── v2.0 (block-structured classifier) ──")
results_v2 = run_prompt(v2.content, test_cases, PROVIDER, MODEL)
for r in results_v2:
    match = "✓" if r["actual_output"].lower() == r["expected_output"].lower() else "✗"
    print(f"  {match} '{r['input'][:40]}...' → '{r['actual_output']}'")

## 3. Evaluation with DeepEval

DeepEval provides research-backed metrics for evaluating LLM outputs. We use:
- **AnswerRelevancyMetric** — Does the output address the input?
- **GEval (Correctness)** — Does the output match the expected result?

Each metric has a **threshold** — the prompt version must pass ALL thresholds to be considered production-ready.

In [ ]:
try:
    from deepeval import evaluate
    from deepeval.test_case import LLMTestCase, SingleTurnParams
    from deepeval.metrics import GEval, AnswerRelevancyMetric

    DEEPEVAL_AVAILABLE = True
    print("DeepEval installed — using research-backed metrics.")
except ImportError:
    DEEPEVAL_AVAILABLE = False
    print("DeepEval not installed — using heuristic fallback.")
    print("Install with: pip install deepeval")

In [ ]:
def evaluate_results(results: list[dict], use_deepeval: bool = True) -> dict:
    """Evaluate prompt outputs with DeepEval or heuristic fallback."""
    if not use_deepeval:
        return _heuristic_evaluate(results)

    # Build metrics
    relevancy = AnswerRelevancyMetric(threshold=0.7)
    correctness = GEval(
        name="Correctness",
        criteria="Determine if the actual output matches the expected output in meaning.",
        evaluation_params=[
            SingleTurnParams.ACTUAL_OUTPUT,
            SingleTurnParams.EXPECTED_OUTPUT,
        ],
        threshold=0.6,
    )

    # Build test cases
    cases = []
    for r in results:
        kwargs = {"input": r["input"], "actual_output": r["actual_output"]}
        if r.get("expected_output"):
            kwargs["expected_output"] = r["expected_output"]
        cases.append(LLMTestCase(**kwargs))

    # Run evaluation
    result = evaluate(test_cases=cases, metrics=[relevancy, correctness])
    return {
        "framework": "deepeval",
        "overall_success": result.success if hasattr(result, "success") else None,
        "test_cases": len(cases),
    }


def _heuristic_evaluate(results: list[dict]) -> dict:
    """Simple heuristic evaluation when DeepEval isn't available."""
    correct = sum(
        1 for r in results
        if r["actual_output"].lower().strip() == r["expected_output"].lower().strip()
    )
    accuracy = correct / len(results) if results else 0
    return {
        "framework": "heuristic",
        "accuracy": round(accuracy, 3),
        "correct": correct,
        "total": len(results),
        "overall_success": accuracy >= 0.6,
    }


# Evaluate both versions
print("Evaluating v1.0...")
eval_v1 = evaluate_results(results_v1, DEEPEVAL_AVAILABLE)
print(f"  Result: {eval_v1}\n")

print("Evaluating v2.0...")
eval_v2 = evaluate_results(results_v2, DEEPEVAL_AVAILABLE)
print(f"  Result: {eval_v2}")

## 4. Comparison Report

Side-by-side comparison of both prompt versions with their evaluation results.

In [ ]:
print("=" * 60)
print("  PROMPT VERSION COMPARISON")
print("=" * 60)
print()

for label, ver, results, ev in [("v1.0", v1, results_v1, eval_v1), ("v2.0", v2, results_v2, eval_v2)]:
    print(f"  {label}: {ver.notes}")
    print(f"  Author: {ver.author}")
    print(f"  Prompt length: {len(ver.content)} chars")
    print(f"  Evaluation: {ev}")
    
    # Show individual results
    correct = sum(1 for r in results if r["actual_output"].lower() == r["expected_output"].lower())
    print(f"  Exact matches: {correct}/{len(results)}")
    print()

# Determine winner
if eval_v1.get("overall_success") and not eval_v2.get("overall_success"):
    winner = "v1.0"
elif eval_v2.get("overall_success") and not eval_v1.get("overall_success"):
    winner = "v2.0"
else:
    # Compare accuracy if both pass or both fail
    acc1 = eval_v1.get("accuracy", 0)
    acc2 = eval_v2.get("accuracy", 0)
    winner = "v2.0" if acc2 >= acc1 else "v1.0"

print("-" * 60)
print(f"  WINNER: {winner}")
print(f"  Recommendation: Promote {winner} to production.")
print("-" * 60)

## 5. Save Artifacts

Export the comparison data as JSON — a visible, savable artifact for portfolio or reporting.

In [ ]:
import json
from pathlib import Path

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

artifact = {
    "prompt": "support-classifier",
    "versions": {
        "v1.0": {
            "content": v1.content,
            "author": v1.author,
            "notes": v1.notes,
            "evaluation": eval_v1,
            "results": results_v1,
        },
        "v2.0": {
            "content": v2.content,
            "author": v2.author,
            "notes": v2.notes,
            "evaluation": eval_v2,
            "results": results_v2,
        },
    },
    "winner": winner,
    "timestamp": datetime.now().isoformat(),
}

artifact_path = output_dir / "prompt_comparison.json"
artifact_path.write_text(json.dumps(artifact, indent=2))
print(f"Saved: {artifact_path}")
print(f"\nArtifact contains:")
print(f"  - Both prompt versions with full content")
print(f"  - Evaluation results for each version")
print(f"  - Individual test case results")
print(f"  - Winner designation")